In [3]:
from chessdotcom import ChessDotComClient
import re
import datetime
import pytz
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [221]:
### Enter Chess.com username to begin ###
client = ChessDotComClient(user_agent = "My Python Application...")
response = 0
while response == 0:
    username = str(input("Enter Username: "))
    try:
        response = client.get_player_profile(username)
        print('Username entered successfully')
    except chessdotcom.errors.ChessDotComClientError:
        print('Invalid Username')

# reassigning the username variable to the value stored on chess.com
url = response.player.url
pattern = r'[^/]+$'
match = re.findall(pattern, url)
username = match[0]

Enter Username:  hikaru


Username entered successfully


In [7]:
tz_list = []
for tz in pytz.all_timezones:
    print(tz)
    tz_list.append(tz)

Africa/Abidjan
Africa/Accra
Africa/Addis_Ababa
Africa/Algiers
Africa/Asmara
Africa/Asmera
Africa/Bamako
Africa/Bangui
Africa/Banjul
Africa/Bissau
Africa/Blantyre
Africa/Brazzaville
Africa/Bujumbura
Africa/Cairo
Africa/Casablanca
Africa/Ceuta
Africa/Conakry
Africa/Dakar
Africa/Dar_es_Salaam
Africa/Djibouti
Africa/Douala
Africa/El_Aaiun
Africa/Freetown
Africa/Gaborone
Africa/Harare
Africa/Johannesburg
Africa/Juba
Africa/Kampala
Africa/Khartoum
Africa/Kigali
Africa/Kinshasa
Africa/Lagos
Africa/Libreville
Africa/Lome
Africa/Luanda
Africa/Lubumbashi
Africa/Lusaka
Africa/Malabo
Africa/Maputo
Africa/Maseru
Africa/Mbabane
Africa/Mogadishu
Africa/Monrovia
Africa/Nairobi
Africa/Ndjamena
Africa/Niamey
Africa/Nouakchott
Africa/Ouagadougou
Africa/Porto-Novo
Africa/Sao_Tome
Africa/Timbuktu
Africa/Tripoli
Africa/Tunis
Africa/Windhoek
America/Adak
America/Anchorage
America/Anguilla
America/Antigua
America/Araguaina
America/Argentina/Buenos_Aires
America/Argentina/Catamarca
America/Argentina/ComodRivad

In [223]:
while True:
    timezone = str(input("Enter Timezone: "))
    if timezone in tz_list:
        print('Timezone entered successfully')
        break
    else:
        print('Invalid Timezone')
    

Enter Timezone:  US/Eastern


Timezone entered successfully


In [225]:
### Get player profile ###

if response.player.name is None:
    name = pd.NA
else: 
    name = response.player.name

if response.player.country is None:
    country = pd.NA
else: 
    response_country = client.get_country_details(response.player.country[-2:])
    country = response_country.country.name

if response.player.joined is None:
    joined = pd.NA
else: 
    joined = datetime.datetime.fromtimestamp(response.player.joined)

if response.player.last_online is None:
    last_online = pd.NA
else: 
    last_online = datetime.datetime.fromtimestamp(response.player.last_online)

profile = {
    "Profile_Info": ["Name", "Country", "Joined", "Last Online"],
    username: [name, country, joined, last_online]}

df = pd.DataFrame(profile)
df.set_index("Profile_Info", inplace=True)
df = df.rename_axis(None)
df

,Hikaru
Name,Hikaru Nakamura
Country,United States
Joined,2014-01-06 16:20:58
Last Online,2024-11-29 03:40:58


In [227]:
### Get all-time player stats ###

response2 = client.get_player_stats(username)

if response2.stats.chess_rapid is None:
    rapid_rating = pd.NA
    rapid_wld = pd.NA
else: 
    rapid_rating = response2.stats.chess_rapid.last.rating
    rapid_wld = f"{str(response2.stats.chess_rapid.record.win)}-{str(response2.stats.chess_rapid.record.loss)}-{str(response2.stats.chess_rapid.record.draw)}"

if response2.stats.chess_blitz is None:
    blitz_rating = pd.NA
    blitz_wld = pd.NA
else: 
    blitz_rating = response2.stats.chess_blitz.last.rating
    blitz_wld = f"{str(response2.stats.chess_blitz.record.win)}-{str(response2.stats.chess_blitz.record.loss)}-{str(response2.stats.chess_blitz.record.draw)}"
    
if response2.stats.chess_bullet is None:
    bullet_rating = pd.NA
    bullet_wld = pd.NA
else: 
    bullet_rating = response2.stats.chess_bullet.last.rating
    bullet_wld = f"{str(response2.stats.chess_bullet.record.win)}-{str(response2.stats.chess_bullet.record.loss)}-{str(response2.stats.chess_bullet.record.draw)}"

if response2.stats.chess_daily is None:
    daily_rating = pd.NA
    daily_wld = pd.NA
else: 
    daily_rating = response2.stats.chess_daily.last.rating
    daily_wld = f"{str(response2.stats.chess_daily.record.win)}-{str(response2.stats.chess_daily.record.loss)}-{str(response2.stats.chess_daily.record.draw)}"

stats = {
    "Profile_Info": ["Rapid Chess Rating", "Rapid Chess Record",
                     "Blitz Chess Rating", "Blitz Chess Record",
                     "Bullet Chess Rating", "Bullet Chess Record",
                     "Daily Chess Rating", "Daily Chess Record"],
    "Username": [rapid_rating, rapid_wld,
                 blitz_rating, blitz_wld,
                 bullet_rating, bullet_wld,
                 daily_rating, daily_wld
                 ]}

df = pd.DataFrame(stats)
df.rename(columns={"Username": username}, inplace=True)
df.set_index("Profile_Info", inplace=True)
df = df.rename_axis(None)
df

,Hikaru
Rapid Chess Rating,2769
Rapid Chess Record,182-53-184
Blitz Chess Rating,3231
Blitz Chess Record,29650-4921-3881
Bullet Chess Rating,3319
Bullet Chess Record,14246-1967-899
Daily Chess Rating,2239
Daily Chess Record,73-11-4


In [229]:
## Make a dataframe of all games the player has ever played ##
response3 = client.get_player_game_archives(username)
months_list = []
for i in range(len(response3.archives)):
    months_list.append(response3.archives[i][-7:])

live_games = []
for i in range(len(months_list)):
    year = int(months_list[i][:4])
    month = int(months_list[i][5:])
    response4 = client.get_player_games_by_month(username, year, month)
    for i in range(len(response4.games)):
        if response4.games[i].pgn and (response4.games[i].pgn[:20] == '[Event "Live Chess"]' or response4.games[i].pgn[:20] == '[Event "Let\'s Play"]'):
            live_games.append(response4.games[i])

In [17]:
def convert_to_timezone(date, timezone):
    date = pd.to_datetime(date)
    date = date.tz_localize('UTC')
    date = str(date.tz_convert(timezone))
    return date[:-9]

In [19]:
def game_info(num):
    if live_games[num].white.username != username:
        opponent = live_games[num].white.username
        my_result = live_games[num].black.result
        opponent_result = live_games[num].white.result
        my_rating = int(live_games[num].black.rating)
        opponent_rating = int(live_games[num].white.rating)
        pieces = 'black'
    else: 
        opponent = live_games[num].black.username
        my_result = live_games[num].white.result
        opponent_result = live_games[num].black.result
        my_rating = int(live_games[num].white.rating)
        opponent_rating = int(live_games[num].black.rating)
        pieces = 'white'
    
    if my_result == 'win':
        result = 'win'
        game_ending = opponent_result
    elif my_result in ['resigned', 'checkmated', 'timeout']:
        result = 'loss'
        game_ending = my_result
    else:
        result = 'draw'
        game_ending = my_result

    list = [pieces, opponent, result, game_ending, my_rating, opponent_rating]
    return list

In [21]:
def time_control(num):
    try:
        time = live_games[num].time_control
        if '+' in time:
            if time[-2] == '+':
                time = str(int(time[:-2])//60) + ' minutes, ' + time[-1] + ' seconds per move'
            else:
                time = str(int(time[:-3])//60) + ' minutes, ' + time[-2:] + ' seconds per move'
        elif live_games[num].time_class == 'daily':
            time = 'daily'
        else:
            time = str(int(time)//60) + ' minutes, ' +  '0 seconds per move'

        if time == '0 minutes, 0 seconds per move':
            time = 'less than 1 minute'
        
        return time
        
    except ValueError:
        time = 'N/A'
        return time

In [233]:
def opening(num):
    url = live_games[num].eco
    pattern1 = r'[^/]+$'
    match1 = re.findall(pattern1, url)
    pattern2 = r'-\d.*$|with-\d.*$|\.{3}.*$'
    match2 = re.sub(pattern2, "", match1[0])
    match3 = re.sub(r'-', " ", match2)
    return match3

In [235]:
# Specify column names
columns = ['Date', 'Played As', 'My Rating', 'Opponent', 'Opponent Rating', 'Result', 'Game Ending', 'Game Type', 'Time Controls', 'Opening', 'URL']

# Create an empty DataFrame
chess_games = pd.DataFrame(columns=columns)

dates = []
played_as = []
my_rating = []
opponent = []
opponent_rating = []
result = []
game_ending = []
game_type = []
controls = []
chess_opening = []
url = []

for i in range(len(live_games)):
    readable_date = datetime.datetime.utcfromtimestamp(live_games[i].end_time)
    dates.append(convert_to_timezone(readable_date, timezone))
    played_as.append(game_info(i)[0])
    opponent.append(game_info(i)[1])
    result.append(game_info(i)[2])
    game_ending.append(game_info(i)[3])
    my_rating.append(game_info(i)[4])
    opponent_rating.append(game_info(i)[5])
    game_type.append(live_games[i].time_class)
    controls.append(time_control(i))
    chess_opening.append(opening(i))
    url.append(live_games[i].url)

chess_games['Date'] = dates
chess_games['Played As'] = played_as
chess_games['My Rating'] = my_rating
chess_games['Opponent'] = opponent
chess_games['Opponent Rating'] = opponent_rating
chess_games['Result'] = result
chess_games['Game Ending'] = game_ending
chess_games['Game Type'] = game_type
chess_games['Time Controls'] = controls
chess_games['Opening'] = chess_opening
chess_games['URL'] = url
chess_games['Year'] = chess_games['Date'].str[:4]
chess_games['Month'] = chess_games['Date'].str[5:7]


In [27]:
def filtered_results(played_as, my_rating_lowerbound, my_rating_upperbound, opponent, opponent_rating_lowerbound, opponent_rating_upperbound, 
                     year, month, result, game_ending, game_type, time_controls, opening):
    global filtered_df
    filtered_df = chess_games
    if played_as != 'ALL':
        filtered_df = filtered_df[filtered_df['Played As'] == played_as]
    if my_rating_lowerbound != 'ALL':
        filtered_df = filtered_df[filtered_df['My Rating'] >= int(my_rating_lowerbound)]
    if my_rating_upperbound != 'ALL':
        filtered_df = filtered_df[filtered_df['My Rating'] <= int(my_rating_upperbound)]
    if opponent != 'ALL':
        filtered_df = filtered_df[filtered_df['Opponent'].str.contains(opponent, case=False, na=False)]
    if opponent_rating_lowerbound != 'ALL':
        filtered_df = filtered_df[filtered_df['Opponent Rating'] >= int(opponent_rating_lowerbound)]
    if opponent_rating_upperbound != 'ALL':
        filtered_df = filtered_df[filtered_df['Opponent Rating'] <= int(opponent_rating_upperbound)]
    if year != 'ALL':
        filtered_df = filtered_df[filtered_df['Year'] == year]
    if month != 'ALL':
        filtered_df = filtered_df[filtered_df['Month'] == month]
    if result != 'ALL':
        filtered_df = filtered_df[filtered_df['Result'] == result]
    if game_ending != 'ALL':
        filtered_df = filtered_df[filtered_df['Game Ending'] == game_ending]
    if game_type != 'ALL':
        filtered_df = filtered_df[filtered_df['Game Type'] == game_type]
    if time_controls != 'ALL':
        filtered_df = filtered_df[filtered_df['Time Controls'] == time_controls]
    if opening != 'ALL':
        filtered_df = filtered_df[filtered_df['Opening'].str.contains(opening, case=False, na=False)]
    filtered_df = filtered_df.reset_index(drop=True)
    return filtered_df.iloc[:, :-2]

In [269]:
## Define your filters.  Specify 'ALL' if you do not want to filter that column. Wildcard search for opponent and opening
played_as = 'ALL'
my_rating_lowerbound = 'ALL'
my_rating_upperbound = 'ALL'
opponent  = 'ALL'
opponent_rating_lowerbound = 'ALL'
opponent_rating_upperbound = 'ALL'
year = 'ALL'
month = 'ALL'
result = 'ALL'
game_ending = 'ALL'
game_type = 'ALL'
time_controls = 'ALL'
opening = 'ALL'

# Calling function to print filtered dataframe
filtered_results(played_as, my_rating_lowerbound, my_rating_upperbound, opponent, opponent_rating_lowerbound, opponent_rating_upperbound,
                 year, month, result, game_ending, game_type, time_controls, opening)

,Date,Played As,My Rating,Opponent,Opponent Rating,Result,Game Ending,Game Type,Time Controls,Opening,URL
0,2014-01-06 18:54,white,2354,Godswill,2167,win,resigned,blitz,"3 minutes, 0 seconds per move",Vienna Game Max Lange Paulsen Variation,https://www.chess.com/game/live/692667823
1,2014-01-06 19:00,black,2438,Godswill,2163,win,timeout,blitz,"3 minutes, 0 seconds per move",Indian Game Accelerated Variation,https://www.chess.com/game/live/692670646
2,2014-01-06 19:35,black,2336,manueliux_06,2104,win,resigned,bullet,"1 minutes, 0 seconds per move",Pirc Defense Modern Defense Geller System,https://www.chess.com/game/live/692695595
3,2014-01-06 19:37,white,2411,manueliux_06,2101,win,resigned,bullet,"1 minutes, 0 seconds per move",Caro Kann Defense Two Knights Attack,https://www.chess.com/game/live/692696567
4,2014-01-06 19:46,white,2509,Dmitriy_From_Russia,2266,win,resigned,blitz,"3 minutes, 0 seconds per move",Kings Indian Defense Fianchetto Variation,https://www.chess.com/game/live/692700868
...,...,...,...,...,...,...,...,...,...,...,...
57598,2024-11-30 12:36,black,3314,KogotBobra17,2734,win,resigned,bullet,"1 minutes, 0 seconds per move",Caro Kann Defense Gurgenidze System,https://www.chess.com/game/live/126727173293
57599,2024-11-30 12:38,white,3318,gurelediz,3111,win,timeout,bullet,"1 minutes, 0 seconds per move",Caro Kann Defense Two Knights Attack,https://www.chess.com/game/live/126727185675
57600,2024-11-30 12:40,black,3319,AlbusSevresPotter,2731,win,resigned,bullet,"1 minutes, 0 seconds per move",Reti Opening Kingside Fianchetto Variation,https://www.chess.com/game/live/126727214495
57601,2024-11-30 12:41,black,3319,Zgorl,2705,win,resigned,bullet,"1 minutes, 0 seconds per move",Queens Pawn Opening,https://www.chess.com/game/live/126727239945


In [219]:
# filtered_df.to_excel('filtered_df.xlsx', index=False)

In [271]:
filtered_df['Result'].value_counts()

Result
win     45490
loss     6958
draw     5155
Name: count, dtype: int64

In [281]:
filtered_df['Opening'].value_counts().head(5)

Opening
Modern Defense                                       2816
Modern Defense Standard Line                         2758
Nimzowitsch Larsen Attack Modern Variation           1903
Reti Opening Nimzowitsch Larsen Attack               1408
Indian Game Knights Variation East Indian Defense    1277
Name: count, dtype: int64

In [275]:
# Group by 'Category' and calculate value counts for 'Subcategory'
grouped = filtered_df.groupby('Opening')['Result'].value_counts().unstack(fill_value=0)

# Calculate win percentage
grouped['win_percentage'] = (grouped['win'] / grouped.sum(axis=1)) * 100
grouped['win_percentage'] = grouped['win_percentage'].round(1)

# Calculate value counts for the 'Category' column
category_counts = filtered_df['Opening'].value_counts()

# Add the value counts as a column in the grouped DataFrame
grouped['count'] = category_counts

# Sort the DataFrame by the value counts
grouped = grouped.loc[category_counts.index]

grouped_reset = grouped.reset_index()
grouped_reset = grouped_reset[['Opening', 'count', 'win_percentage']]
print(grouped_reset.to_string(index=False))


                                                                                     Opening  count  win_percentage
                                                                             Modern Defense    2816            83.0
                                                                Modern Defense Standard Line   2758            81.5
                                                  Nimzowitsch Larsen Attack Modern Variation   1903            79.9
                                                      Reti Opening Nimzowitsch Larsen Attack   1408            80.9
                                           Indian Game Knights Variation East Indian Defense   1277            82.5
                                                   Closed Sicilian Defense Grand Prix Attack    905            87.4
                                                  Reti Opening Kingside Fianchetto Variation    895            86.5
                                                         Reti Opening Ni

In [277]:
filtered_df['Time Controls'].value_counts()

Time Controls
3 minutes, 0 seconds per move      34643
1 minutes, 0 seconds per move      16410
3 minutes, 1 seconds per move       3690
3 minutes, 2 seconds per move        500
1 minutes, 1 seconds per move        410
5 minutes, 0 seconds per move        315
5 minutes, 1 seconds per move        282
10 minutes, 0 seconds per move       264
10 minutes, 2 seconds per move       219
daily                                175
10 minutes, 5 seconds per move       119
5 minutes, 2 seconds per move        107
60 minutes, 45 seconds per move       73
less than 1 minute                    72
15 minutes, 2 seconds per move        64
2 minutes, 0 seconds per move         62
20 minutes, 10 seconds per move       59
15 minutes, 3 seconds per move        48
5 minutes, 5 seconds per move         36
15 minutes, 0 seconds per move        13
25 minutes, 10 seconds per move       11
15 minutes, 10 seconds per move        9
10 minutes, 10 seconds per move        7
2 minutes, 1 seconds per move          6
10

In [279]:
filtered_df['Game Type'].value_counts()

Game Type
blitz     39654
bullet    16960
rapid       814
daily       175
Name: count, dtype: int64